In [1]:
from sage.all import *

class Paillier:
    
    def __init__(self, bits=1024):
        """
        Génération des clés. 
        bits : taille des nombres premiers p et q.
        """
        # 1. Choisir deux grands nombres premiers distincts avec next_prime
        self.p = random_prime(2**bits, lbound=2**(bits-1))
        self.q = random_prime(2**bits, lbound=2**(bits-1))
        
        while self.p == self.q:
            self.q = random_prime(2**bits, lbound=2**(bits-1))
            
        # 2. Calculer n et n^2
        self.n = self.p * self.q
        self.n2 = self.n**2

        # 3. Paramètre g (souvent n + 1)
        self.g = self.n + 1
        
        # 4. Calcul de Lambda (λ)
        # λ = ppcm(p-1, q-1)
        self.lam = lcm(self.p - 1, self.q - 1)
        
        # Calcul mu en utilisant la méthode interne _L
        # mu = (L(g^lambda mod n^2))^-1 mod n
        u = power_mod(self.g, self.lam, self.n2)
        self.mu = inverse_mod(self._L(u), self.n)

        print("[+] Paillier keys generated, n =", self.n.nbits(), "bits")

    # Fonction L
    def _L(self, u):
        return (u - 1) // self.n

    # Chiffrement
    def encrypt(self, m):
        """
        Chiffre un message m (doit être < n)
        """

        m = ZZ(m)
        if m < 0 or m >= self.n:
            raise ValueError("Message m doit être dans Z_n")
            
        # Choisir un nombre aléatoire r dans Z*_n
        r = ZZ.random_element(1, self.n)
        while gcd(r, self.n) != 1:
            r = ZZ.random_element(1, self.n)
                        
        # Formule :  Enc(m) = c = g^m * r^n mod n^2
        c = (power_mod(self.g, m, self.n2) * power_mod(r, self.n, self.n2)) % self.n2
        return c


    def decrypt(self, c):
        c = ZZ(c)
        u = power_mod(c, self.lam, self.n2)
        m = (self._L(u) * self.mu) % self.n
        return m


# Enc(m1 + m2)
def paillier_homomorphic_add(crypto, c1, c2):
    return (c1 * c2) % crypto.n2


# Enc(m * k)
def paillier_homomorphic_scalar(crypto, c, k):
    return power_mod(c, k, crypto.n2)
    
# ---------------------------
# TESTS SIMPLES
# ---------------------------
crypto = Paillier(bits=128)

# Test déchiffrement simple
m = 12345
c = crypto.encrypt(m)
m2 = crypto.decrypt(c)
print("Message =", m)
print("Chiffré =", c)
print("Déchiffré =", m2)
assert m == m2, "❌ Erreur: déchiffrement incorrect"

# Test homomorphisme additif
m1 = 10
m2 = 20
c1 = crypto.encrypt(m1)
c2 = crypto.encrypt(m2)
c_sum = (c1 * c2) % crypto.n2
m_sum = crypto.decrypt(c_sum)
print("Homomorphisme: 10+20 =", m_sum)
assert m_sum == (m1 + m2) % crypto.n, "❌ Erreur: homomorphisme incorrect"


# Test homomorphisme additif
m1, m2 = 7, 9
c1 = crypto.encrypt(m1)
c2 = crypto.encrypt(m2)
c_add = paillier_homomorphic_add(crypto, c1, c2)
print("Add decrypt =", crypto.decrypt(c_add))

# Test multiplication scalaire
k = 5
c_scalar = paillier_homomorphic_scalar(crypto, c1, k)
print("Scalar decrypt =", crypto.decrypt(c_scalar))

print("✅ Tous les tests Paillier réussis")


[+] Paillier keys generated, n = 256 bits
Message = 12345
Chiffré = 5906655201276208320906809305555245371547948725611627080510407832361679839066838965470011475923735249737541087385925675823087970463982555587265093401896123
Déchiffré = 12345
Homomorphisme: 10+20 = 30
Add decrypt = 16
Scalar decrypt = 35
✅ Tous les tests Paillier réussis
